# 02. S2Vec식 문맥 임베딩과 다운스트림 평가

목표: 합성 도시 격자에서 셀 자체 피처와 주변 문맥 피처를 만들고, 다운스트림 회귀 성능을 비교합니다.

실행 방법:

```bash
python -m pip install -r s2vec-geospatial-embeddings/requirements.txt
```

이 실습은 논문의 실제 수치를 재현하지 않습니다. 대신 S2Vec가 왜 주변 맥락과 geographic adaptation 평가를 중요하게 보는지 작은 실험으로 확인합니다.

## 1. 여러 부모 셀 합성하기

부모 셀 하나는 16x16 패치로 구성됩니다. 각 부모 셀에는 큰 위치 좌표가 있고, 각 패치에는 작은 위치 좌표가 있습니다. 이렇게 만들면 무작위 분할과 지역 holdout 분할을 모두 실험할 수 있습니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(21)

N_PARENTS = 140
GRID = 16
FEATURES = ['food', 'retail', 'office', 'transit', 'roads', 'parks', 'housing', 'industrial']
F = len(FEATURES)

urban_profile = np.array([7, 7, 9, 6, 7, 1, 6, 1], dtype=float)
suburb_profile = np.array([3, 4, 2, 2, 5, 4, 8, 1], dtype=float)
green_profile = np.array([1, 1, 0.2, 0.4, 1, 10, 1, 0.2], dtype=float)
industrial_profile = np.array([1, 1, 2, 1, 5, 0.5, 1, 8], dtype=float)

def make_parent_cell(parent_x, parent_y):
    cube = np.zeros((GRID, GRID, F), dtype=float)
    # parent_x와 parent_y는 큰 지도에서의 위치를 흉내 냅니다.
    # 특정 권역은 더 도시적이고, 특정 권역은 더 녹지가 많도록 만듭니다.
    metro_strength = np.exp(-((parent_x - 0.42) ** 2 + (parent_y - 0.55) ** 2) / 0.08)
    green_strength = np.exp(-((parent_x - 0.78) ** 2 + (parent_y - 0.30) ** 2) / 0.05)
    industrial_strength = np.exp(-((parent_x - 0.25) ** 2 + (parent_y - 0.20) ** 2) / 0.05)

    for r in range(GRID):
        for c in range(GRID):
            local_x = c / (GRID - 1)
            local_y = r / (GRID - 1)
            center_bump = np.exp(-((local_x - 0.5) ** 2 + (local_y - 0.5) ** 2) / 0.06)
            edge_green = np.exp(-((local_x - 0.12) ** 2 + (local_y - 0.82) ** 2) / 0.05)
            industrial_band = np.exp(-((local_y - 0.20) ** 2) / 0.02)

            lam = 0.25
            lam += (0.30 + metro_strength) * center_bump * urban_profile
            lam += (0.60 - 0.25 * metro_strength) * suburb_profile
            lam += (0.25 + green_strength) * edge_green * green_profile
            lam += (0.15 + industrial_strength) * industrial_band * industrial_profile
            cube[r, c] = rng.poisson(np.clip(lam, 0.05, None))
    return cube

parent_xy = rng.random((N_PARENTS, 2))
cubes = np.stack([make_parent_cell(x, y) for x, y in parent_xy])

print('parent images:', cubes.shape)
print('one parent x/y:', parent_xy[0])

## 2. 문맥 피처 만들기

S2Vec의 실제 encoder는 Transformer self-attention을 사용합니다. 여기서는 단순한 3x3 평균과 부모 셀 평균을 문맥 피처로 추가합니다. 중요한 점은 셀 자체 카운트만 보는 모델과 주변 구조까지 보는 모델을 분리해 비교하는 것입니다.

In [ ]:
def mean_pool_3x3(batch):
    padded = np.pad(batch, ((0, 0), (1, 1), (1, 1), (0, 0)), mode='edge')
    pooled = np.zeros_like(batch, dtype=float)
    for dr in range(3):
        for dc in range(3):
            pooled += padded[:, dr:dr + GRID, dc:dc + GRID, :]
    return pooled / 9.0

local_context = mean_pool_3x3(cubes)
parent_mean = cubes.mean(axis=(1, 2))

raw_X = cubes.reshape(-1, F)
context_X = local_context.reshape(-1, F)
parent_context_X = np.repeat(parent_mean, GRID * GRID, axis=0)

patch_xy_one = np.array([(c / (GRID - 1), r / (GRID - 1)) for r in range(GRID) for c in range(GRID)])
patch_xy = np.tile(patch_xy_one, (N_PARENTS, 1))
geo_xy = np.repeat(parent_xy, GRID * GRID, axis=0)

# raw 모델은 현재 셀 카운트만 봅니다.
X_raw = raw_X

# context 모델은 현재 셀, 주변 3x3 평균, 부모 셀 평균, 좌표 힌트를 함께 봅니다.
# 좌표 힌트는 논문에서 Space2Vec 같은 location signal을 붙이는 변형과 비슷한 역할입니다.
X_contextual = np.concatenate([raw_X, context_X, parent_context_X, patch_xy, geo_xy], axis=1)

print('raw features:', X_raw.shape)
print('contextual features:', X_contextual.shape)

## 3. 합성 타깃 만들기

인구 밀도와 중위 소득은 셀 자체보다 주변 상업/교통/주거 패턴에 영향을 받도록 설계합니다. 이렇게 해야 문맥 표현의 이점을 관찰할 수 있습니다.

In [ ]:
idx = {name: i for i, name in enumerate(FEATURES)}

population = (
    0.18 * context_X[:, idx['roads']]
    + 0.16 * context_X[:, idx['transit']]
    + 0.12 * raw_X[:, idx['housing']]
    + 0.10 * context_X[:, idx['food']]
    - 0.05 * raw_X[:, idx['parks']]
    + rng.normal(0, 0.8, size=len(raw_X))
)

income = (
    0.20 * geo_xy[:, 0]
    + 0.15 * geo_xy[:, 1]
    + 0.12 * context_X[:, idx['office']]
    + 0.08 * context_X[:, idx['retail']]
    - 0.06 * raw_X[:, idx['industrial']]
    + rng.normal(0, 0.5, size=len(raw_X))
)

def minmax(values):
    values = np.asarray(values, dtype=float)
    return (values - values.min()) / (values.max() - values.min() + 1e-9)

y_population = minmax(population)
y_income = minmax(income)

print('population range:', y_population.min(), y_population.max())
print('income range:', y_income.min(), y_income.max())

## 4. Ridge 회귀 평가 함수

논문은 2-layer MLP를 사용하지만, 여기서는 수식이 보이는 ridge regression을 씁니다. 임베딩 비교의 핵심은 모델을 복잡하게 만드는 것이 아니라 입력 표현의 차이를 보는 것입니다.

In [ ]:
def standardize_by_train(X, train_mask):
    mean = X[train_mask].mean(axis=0)
    std = X[train_mask].std(axis=0) + 1e-6
    return (X - mean) / std

def fit_ridge(X, y, alpha=3.0):
    Xb = np.column_stack([np.ones(len(X)), X])
    penalty = np.eye(Xb.shape[1]) * alpha
    penalty[0, 0] = 0.0
    return np.linalg.solve(Xb.T @ Xb + penalty, Xb.T @ y)

def predict_ridge(X, weights):
    Xb = np.column_stack([np.ones(len(X)), X])
    return Xb @ weights

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2) + 1e-12
    return 1 - ss_res / ss_tot

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def evaluate(X, y, train_mask, test_mask):
    Xs = standardize_by_train(X, train_mask)
    w = fit_ridge(Xs[train_mask], y[train_mask])
    pred = predict_ridge(Xs[test_mask], w)
    return r2_score(y[test_mask], pred), mae(y[test_mask], pred)

## 5. 무작위 분할과 지리적 holdout 비교

무작위 분할은 가까운 지역이 train/test에 동시에 들어갈 수 있습니다. 지리적 holdout은 `parent_x > 0.72`인 동쪽 권역 전체를 테스트로 남깁니다. 논문의 zero-shot geographic adaptation과 같은 평가 철학입니다.

In [ ]:
n = len(raw_X)
order = rng.permutation(n)
random_train = np.zeros(n, dtype=bool)
random_test = np.zeros(n, dtype=bool)
random_train[order[: int(0.70 * n)]] = True
random_test[order[int(0.70 * n):]] = True

geo_test = geo_xy[:, 0] > 0.72
geo_train = ~geo_test

experiments = [
    ('random split', random_train, random_test),
    ('geographic holdout', geo_train, geo_test),
]

feature_sets = [
    ('raw counts only', X_raw),
    ('contextual S2-like', X_contextual),
]

targets = [
    ('population', y_population),
    ('income', y_income),
]

print('{:<20} {:<12} {:<20} {:>7} {:>7}'.format('split', 'target', 'features', 'R2', 'MAE'))
print('-' * 72)
for split_name, train_mask, test_mask in experiments:
    for target_name, y in targets:
        for feature_name, X in feature_sets:
            score, error = evaluate(X, y, train_mask, test_mask)
            print(f'{split_name:<20} {target_name:<12} {feature_name:<20} {score:7.3f} {error:7.3f}')

## 6. 예측 결과 시각화

아래는 지리적 holdout에서 인구 밀도를 예측한 결과입니다. 한 점은 하나의 patch cell입니다.

In [ ]:
Xs = standardize_by_train(X_contextual, geo_train)
w = fit_ridge(Xs[geo_train], y_population[geo_train])
pred = predict_ridge(Xs[geo_test], w)

plt.figure(figsize=(5, 5))
plt.scatter(y_population[geo_test], pred, s=8, alpha=0.35)
plt.plot([0, 1], [0, 1], color='black', linewidth=1)
plt.xlabel('true normalized population')
plt.ylabel('predicted')
plt.title('geographic holdout: contextual features')
plt.show()

## 해석

- 문맥 피처가 raw count보다 좋아지는 과제라면, 셀 자체보다 주변 구조가 타깃에 중요하다는 뜻입니다.
- 무작위 분할 성능만 믿으면 공간적으로 가까운 데이터 누수를 과소평가할 수 있습니다.
- 지리적 holdout은 더 어렵지만, 실제 새로운 지역에 배포할 모델을 평가하는 데 더 의미가 있습니다.
- 다음 노트북에서는 마스킹 복원과 멀티모달 결합을 더 직접적으로 실험합니다.